In [2]:
# import dataset 
import pandas as pd 
import numpy as np

# view results 
df = pd.read_csv('fraud_risk_dataset.csv')
df.head()

,transaction_id,customer_age,account_tenure_months,region,transaction_amount,merchant_category,payment_method,device_type,num_transactions_24h,international,previous_chargebacks,fraud
0,100000,65,68,South,130.13,Travel,Digital Wallet,Tablet,4,0,0,0
1,100001,22,4,South,28.27,Travel,Debit,Tablet,1,0,0,0
2,100002,43,53,South,304.83,Travel,Credit,Tablet,2,0,0,0
3,100003,72,69,West,39.87,Restaurants,Debit,Tablet,1,0,1,1
4,100004,21,8,South,125.73,Gas,Debit,Desktop,3,0,0,0


In [3]:
# Feature engineering (account maturity)

# Account Maturity x Pevious chargebacks 
df['tenure_x_chargebacks'] = (df['account_tenure_months'] * df['previous_chargebacks'])

# Account Maturity x velocity 
df['tenure_x_velocity'] = (df['account_tenure_months'] / df['num_transactions_24h'])

# Account Maturity x Transaction Amount 
df['tenure_x_velocity']  = (df['num_transactions_24h'] * df['account_tenure_months'])



In [4]:
# feature enginering (reputation)

# Reputation vs Region 
df['chargeback_x_region'] = (df['previous_chargebacks'] * df['region'])

# Reputation vs Device 
df['chargeback_x_device'] = (df['previous_chargebacks'] * df['device_type'])

# Reputation vs Customer age 
df['chargebacks_x_age'] = (df['previous_chargebacks'] * df['customer_age'])

# Reputations vs International 
df['chargebacks_x_international'] = (df['previous_chargebacks'] * df['international'])


In [5]:
# feature engineering (patterns and risk)

df['youth_risk'] = (df['customer_age'] < 30) & (df['num_transactions_24h'] >= 5 )

df['high_foreign_amount'] = (df['transaction_amount'] > 350 ) & (df['international'] == 1)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


X = df.drop(columns = ['transaction_id', 'fraud'])
y = df['fraud']


#auto detect categorical vs numerical columns 
num_col = X.select_dtypes(include = ['number']).columns.to_list()
cat_col = X.select_dtypes(include = ['object']).columns.to_list()

# preprocesser
prep = ColumnTransformer(transformers = [
    ('num_col', 'passthrough', num_col),
    ('cat_col', OneHotEncoder(handle_unknown= 'ignore'),
     cat_col)])


In [7]:
# Time to train the data

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.30,
                                                    stratify= y, random_state= 42)



In [8]:

# fit interaction features into pipeline
from Feature_eng import add_interactions
fe_prep = FunctionTransformer(add_interactions, validate=False)

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline(steps= [
                ('fe', fe_prep),
                ('prep', prep),
                ('model', RandomForestClassifier(
                    n_estimators= 300, # how many trees model will build 
                    max_depth = None, # How deep each tree can grow (model complexity)
                    min_samples_split= 10, # Minimun number of samples in each split
                    min_samples_leaf = 5, # Minimun number of leafs in each tree
                    class_weight= 'balanced', # balance terget variable 
                    n_jobs = -1, # Use all cpu for faster training 
                    random_state = 42, # Ensures reproducitbity
))])



rf.fit(X_train, y_train)



Pipeline(steps=[('fe',
                 FunctionTransformer(func=<function add_interactions at 0x00000234430028E0>)),
                ('prep',
                 ColumnTransformer(transformers=[('num_col', 'passthrough',
                                                  ['customer_age',
                                                   'account_tenure_months',
                                                   'transaction_amount',
                                                   'num_transactions_24h',
                                                   'international',
                                                   'previous_chargebacks',
                                                   'tenure_x_chargebacks',
                                                   'tenure_x_velocity',
                                                   'chargebacks_x_age',
                                                   'chargebacks_x_international']),
                                                 ('cat_col',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['region',
                                                   'merchant_category',
                                                   'payment_method',
                                                   'device_type',
                                                   'chargeback_x_region',
                                                   'chargeback_x_device'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        min_samples_leaf=5,
                                        min_samples_split=10, n_estimators=300,
                                        n_jobs=-1, random_state=42))])

In [10]:
# Let's view the classification report for the model and determine if we need threshold tunning or not
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
rf_pred = rf.predict(X_test) # Prediction 

print(f'classification_report{classification_report(y_test, rf_pred)}') #results

classification_report              precision    recall  f1-score   support

           0       0.94      0.95      0.94      1391
           1       0.25      0.21      0.23       109

    accuracy                           0.90      1500
   macro avg       0.59      0.58      0.59      1500
weighted avg       0.89      0.90      0.89      1500



In [11]:
# Let's view results with threshold tunning invovled 

rf_pred = rf.predict_proba(X_test)[:, 1] # Probability prediction (No binanry output)
thresholds = [0.15, 0.20, 0.25, 0.27, 0.30] # different thresholds 

for t in thresholds:
    proba = (rf_pred >= t).astype(int)
    
    print(f' Thresholds {t}')
    print(f'roc auc score {roc_auc_score(y_test, proba)}')
    print(f' Classification Report {classification_report(y_test, proba)}')

 Thresholds 0.15
roc auc score 0.6251953910789545
 Classification Report               precision    recall  f1-score   support

           0       0.97      0.42      0.58      1391
           1       0.10      0.83      0.18       109

    accuracy                           0.45      1500
   macro avg       0.54      0.63      0.38      1500
weighted avg       0.91      0.45      0.55      1500

 Thresholds 0.2
roc auc score 0.6690850091347391
 Classification Report               precision    recall  f1-score   support

           0       0.97      0.62      0.76      1391
           1       0.13      0.72      0.22       109

    accuracy                           0.63      1500
   macro avg       0.55      0.67      0.49      1500
weighted avg       0.90      0.63      0.72      1500

 Thresholds 0.25
roc auc score 0.7188610926071272
 Classification Report               precision    recall  f1-score   support

           0       0.97      0.74      0.84      1391
           1       

In [12]:
# Moving forward let's hyper tune the model utilzing the 0.28 threshold. This so far has produced the best results in eval. 

In [13]:
# Let's being modeling for RandomSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, roc_auc_score

param_dict = {
    'model__n_estimators': [100, 200, 300, 400, 500], # Number of trees (more trees = smoother predictions)
    'model__max_depth': [None, 3, 5, 6, 10,], # How deep each tree can grow (smaller depth -> fewer splits = less overfitting)
    'model__min_samples_leaf': [2, 4, 6, 8, 10], # Minimum number of samples before each split at a node (higher = more conservative)
    'model__min_samples_split': [2, 4, 5, 10], # Minimum number of samples in each split (forces generalization)
    'model__max_features': ['sqrt', 'log2', 0.5] # adjusting the allowed features per each split (forces randomness = better generalization)
}
rf_tuner = RandomizedSearchCV(
    estimator = rf, # Model
    param_distributions = param_dict, # the knobs model is allowed to adjust
    n_iter = 20, # Amount of iterations model is allowed to perform
    scoring = 'roc_auc', # Optimizing for evalutation metric 
    cv = 5, # Split data in 5 cross vals
    verbose = 1, # Demonstrate what you are doing 
    n_jobs = -1, # Use all of the cpu's for faster training 
    random_state = 42 # ensure reproducibiltiy
)
    
rf_tuner.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('fe',
                                              FunctionTransformer(func=<function add_interactions at 0x00000234430028E0>)),
                                             ('prep',
                                              ColumnTransformer(transformers=[('num_col',
                                                                               'passthrough',
                                                                               ['customer_age',
                                                                                'account_tenure_months',
                                                                                'transaction_amount',
                                                                                'num_transactions_24h',
                                                                                'international',
                                                                                'previous_chargebacks',
                                                                                'tenure_x_chargebacks',
                                                                                'tenure_x...
                                                                     min_samples_split=10,
                                                                     n_estimators=300,
                                                                     n_jobs=-1,
                                                                     random_state=42))]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'model__max_depth': [None, 3, 5, 6, 10],
                                        'model__max_features': ['sqrt', 'log2',
                                                                0.5],
                                        'model__min_samples_leaf': [2, 4, 6, 8,
                                                                    10],
                                        'model__min_samples_split': [2, 4, 5,
                                                                     10],
                                        'model__n_estimators': [100, 200, 300,
                                                                400, 500]},
                   random_state=42, scoring='roc_auc', verbose=1)

In [14]:
# Evaluate the best param results 
best_score = rf_tuner.best_score_
best_param = rf_tuner.best_params_ 
best_estimator = rf_tuner.best_estimator_
# Print
print(f'-- Best score --- {best_score}')
print(f'-- Best Parameters --- {best_param}')
print(f'--- Best Estimators --- {best_estimator}')

-- Best score --- 0.7308219206250711
-- Best Parameters --- {'model__n_estimators': 200, 'model__min_samples_split': 5, 'model__min_samples_leaf': 6, 'model__max_features': 'log2', 'model__max_depth': 3}
--- Best Estimators --- Pipeline(steps=[('fe',
                 FunctionTransformer(func=<function add_interactions at 0x00000234430028E0>)),
                ('prep',
                 ColumnTransformer(transformers=[('num_col', 'passthrough',
                                                  ['customer_age',
                                                   'account_tenure_months',
                                                   'transaction_amount',
                                                   'num_transactions_24h',
                                                   'international',
                                                   'previous_chargebacks',
                                                   'tenure_x_chargebacks',
                                            

In [15]:
y_prob = best_estimator.predict_proba(X_test)[:,1]
print('score', roc_auc_score(y_test, y_prob)) 


score 0.7195074495940483


In [16]:
# The best iteration of parameters optimized on ROC AUC score at 72%. 
# These results show that the model can tell a difference between the two classes. 

In [17]:
# Let's find the best probablity cuttoff 
from sklearn.metrics import precision_recall_curve

# for each threshold calculate precision, recall and store it as an array 
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

# Compute f1_score for each threshold 
f1_score = 2 * (precision * recall) / (precision + recall + 1e-10)

# find index of best f1
best_idx = np.argmax(f1_score[:-1])

#Extract the best threshold 
best_thresh = thresholds[best_idx]

# extrac metrics at best thresholds 
best_recall = recall[best_idx]
best_precision = precision[best_idx]
best_f1 = f1_score[best_idx]

In [18]:
# Print best results 
print('Best F1 threshold RF', best_thresh)
print('Best f1', best_f1)
print('precision @ best f1', best_precision)
print('Recall @ best f1', best_recall)

Best F1 threshold RF 0.5129354849374785
Best f1 0.32240437154287527
precision @ best f1 0.22957198443579765
Recall @ best f1 0.5412844036697247


In [19]:
# Lets evaluate the probability cutt off using the best threshold 
y_pred = (y_prob >= best_thresh).astype(int) # Turn output into integer

print(f'classification_report {classification_report(y_test, y_pred)}')
print(f'confusion_matrix {confusion_matrix(y_test, y_pred)}')
print(f'roc auc {roc_auc_score(y_test, y_pred)}')

classification_report               precision    recall  f1-score   support

           0       0.96      0.86      0.91      1391
           1       0.23      0.54      0.32       109

    accuracy                           0.83      1500
   macro avg       0.59      0.70      0.61      1500
weighted avg       0.91      0.83      0.86      1500

confusion_matrix [[1193  198]
 [  50   59]]
roc auc 0.6994703829994922


In [20]:
# Feature importance 

feature_names = best_estimator.named_steps['prep'].get_feature_names_out() # Pull feature names from after preprocessing 
importances = best_estimator.named_steps['model'].feature_importances_ # display the most important features model used to determine predicitions with 

fi = pd.DataFrame({
    'feature': feature_names,
    'importances': importances
}).sort_values('importances', ascending = False)

In [21]:
# View top 5 most important features 

fi.head()

,feature,importances
4,num_col__international,0.139004
2,num_col__transaction_amount,0.121043
5,num_col__previous_chargebacks,0.100541
8,num_col__chargebacks_x_age,0.096714
46,cat_col__chargeback_x_device_DesktopDesktop,0.051382


In [25]:
import joblib 

joblib.dump(best_estimator, 'best_model')

['best_model']